<a href="https://colab.research.google.com/github/w02ya/Job_hub/blob/main/11%EC%9B%94_%EB%B6%84%EC%84%9D_%EC%8B%9C%EA%B0%81%ED%99%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1번 쿼리 - 전체 이벤트 기준 퍼널 분석

In [31]:
# 1. 설치 및 인증
!pip install google-cloud-bigquery plotly pandas --quiet

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import plotly.graph_objects as go

PROJECT_ID = 'gen-lang-client-0420777856'
client = bigquery.Client(project=PROJECT_ID)

In [32]:
# 2. 쿼리 실행
query = """
SELECT
    COUNTIF(event_type = 'view')     AS view_count,
    COUNTIF(event_type = 'cart')     AS cart_count,
    COUNTIF(event_type = 'purchase') AS purchase_count
FROM `gen-lang-client-0420777856.ecommerce_project.events`
"""

df = client.query(query).to_dataframe()

view_count     = df['view_count'][0]
cart_count     = df['cart_count'][0]
purchase_count = df['purchase_count'][0]

view_to_cart     = round(cart_count     / view_count     * 100, 2)
cart_to_purchase = round(purchase_count / cart_count     * 100, 2)

print(f'view → cart 전환율:     {view_to_cart}%')
print(f'cart → purchase 전환율: {cart_to_purchase}%')

view → cart 전환율:     4.77%
cart → purchase 전환율: 30.27%


In [85]:
# 3. 퍼널 차트
stages = ['View', 'Cart', 'Purchase']
values = [view_count, cart_count, purchase_count]

fig = go.Figure()

# 막대 (수치는 위에)
fig.add_trace(go.Bar(
    x=stages,
    y=values,
    text=[f'{view_count:,}', f'{cart_count:,}', f'{purchase_count:,}'],
    textposition='outside',
    marker=dict(color=colors)
))

# 퍼센트만 안에
fig.add_trace(go.Bar(
    x=stages,
    y=values,
    text=['', f'{view_to_cart}%', f'{cart_to_purchase}%'],
    textposition='inside',
    insidetextanchor='middle',
    marker=dict(color=['rgba(0,0,0,0)', 'rgba(0,0,0,0)', 'rgba(0,0,0,0)']),  # 투명
    showlegend=False
))

fig.update_layout(
    title='이커머스 구매 전환 퍼널 (11월 전체)',
    yaxis=dict(type='log', title='Event Count (log scale)'),
    barmode='overlay',    # ← 두 트레이스를 겹치게
    plot_bgcolor='white',
    height=500
)

fig.update_layout(
    title='이커머스 구매 전환 퍼널 - 이벤트 기준 (11월 전체)',
    yaxis=dict(
        type='log',
        title='Event Count (log scale)',
        tickvals=[100000, 1000000, 10000000, 100000000],
        ticktext=['100K', '1M', '10M', '100M']   # ← 눈금 직접 지정
    ),
    barmode='overlay',
    plot_bgcolor='white',
    showlegend=False,    # ← trace 0 레전드도 제거
    height=500
)

fig.show()

2번쿼리 - 유저 기준 퍼널 차트

In [34]:
# 쿼리 실행
query_users = """
WITH funnel_base AS (
    SELECT
        COUNT(DISTINCT CASE WHEN event_type = 'view'     THEN user_id END) AS view_users,
        COUNT(DISTINCT CASE WHEN event_type = 'cart'     THEN user_id END) AS cart_users,
        COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS purchase_users
    FROM `gen-lang-client-0420777856.ecommerce_project.events`
)
SELECT
    view_users,
    cart_users,
    purchase_users,
    ROUND(SAFE_DIVIDE(cart_users,     view_users)     * 100, 2) AS view_to_cart_rate_pct,
    ROUND(SAFE_DIVIDE(purchase_users, cart_users)     * 100, 2) AS cart_to_purchase_rate_pct,
    ROUND(SAFE_DIVIDE(purchase_users, view_users)     * 100, 2) AS final_conversion_rate_pct
FROM funnel_base
"""

df_users = client.query(query_users).to_dataframe()

view_users     = df_users['view_users'][0]
cart_users     = df_users['cart_users'][0]
purchase_users = df_users['purchase_users'][0]
view_to_cart   = df_users['view_to_cart_rate_pct'][0]
cart_to_purch  = df_users['cart_to_purchase_rate_pct'][0]

In [35]:
# 시각화 (방금이랑 동일)
stages = ['View', 'Cart', 'Purchase']
values = [view_users, cart_users, purchase_users]
colors = ['#378ADD', '#1D9E75', '#D4537E']

fig = go.Figure()

fig.add_trace(go.Bar(
    x=stages,
    y=values,
    text=[f'{view_users:,}', f'{cart_users:,}', f'{purchase_users:,}'],
    textposition='outside',
    marker=dict(color=colors)
))

fig.add_trace(go.Bar(
    x=stages,
    y=values,
    text=['', f'{view_to_cart}%', f'{cart_to_purch}%'],
    textposition='inside',
    insidetextanchor='middle',
    marker=dict(color=['rgba(0,0,0,0)','rgba(0,0,0,0)','rgba(0,0,0,0)']),
    showlegend=False
))

fig.update_layout(
    title='이커머스 구매 전환 퍼널 - 유저 기준 (11월 전체)',
    yaxis=dict(
        type='log',
        title='User Count (log scale)',
        tickvals=[10000, 100000, 1000000, 10000000],
        ticktext=['10K', '100K', '1M', '10M']
    ),
    barmode='overlay',
    plot_bgcolor='white',
    showlegend=False,
    height=500
)

fig.show()

3번 쿼리 - 코호별 주차 리텐셜

In [36]:
# 쿼리 실행
query_cohort = """
WITH first_visit AS (
    SELECT
        user_id,
        DATE_TRUNC(DATE(MIN(event_time)), WEEK) AS cohort_week
    FROM `gen-lang-client-0420777856.ecommerce_project.events`
    GROUP BY user_id
),
activity AS (
    SELECT DISTINCT
        user_id,
        DATE(event_time) AS activity_date
    FROM `gen-lang-client-0420777856.ecommerce_project.events`
)
SELECT
    f.cohort_week,
    DATE_DIFF(a.activity_date, f.cohort_week, WEEK) AS week_diff,
    COUNT(DISTINCT a.user_id) AS active_users
FROM first_visit AS f
JOIN activity a ON f.user_id = a.user_id
GROUP BY 1, 2
ORDER BY 1, 2
"""

df_cohort = client.query(query_cohort).to_dataframe()

In [37]:
import plotly.graph_objects as go
import pandas as pd

# 피벗 테이블로 변환
df_pivot = df_cohort.pivot(index='cohort_week', columns='week_diff', values='active_users')

# 코호트 첫 주(week_diff=0) 대비 리텐션 비율(%)로 변환
df_retention = df_pivot.div(df_pivot[0], axis=0) * 100
df_retention = df_retention.round(1)

df_retention = df_retention.fillna(0)

# float으로 변환 (혹시 모를 타입 문제 방지)
df_retention = df_retention.astype(float)

# 히트맵 시각화
fig = go.Figure(go.Heatmap(
    z=df_retention.values,
    x=[f'Week {i}' for i in df_retention.columns],
    y=[str(d) for d in df_retention.index],
    colorscale='Blues',
    text=df_retention.values,
    texttemplate='%{text}%',
    textfont=dict(size=11),
    hoverongaps=False,
    colorbar=dict(title='리텐션 %')
))

fig.update_layout(
    title='코호트별 주차 리텐션 히트맵',
    xaxis=dict(title='경과 주차 (Week Diff)'),
    yaxis=dict(title='코호트 주차 (첫 방문 주)', autorange='reversed'),
    height=400,
    plot_bgcolor='white'
)

fig.show()

4. 주차별 신규 결제 고객수

In [38]:
query_new_customers = """
WITH user_first_purchase AS (
    SELECT
        user_id,
        MIN(event_time) AS first_purchase_time
    FROM `gen-lang-client-0420777856.ecommerce_project.events`
    WHERE event_type = 'purchase'
    GROUP BY user_id
)
SELECT
    DATE_TRUNC(DATE(first_purchase_time), WEEK) AS first_purchase_week,
    COUNT(DISTINCT user_id) AS new_paying_customers
FROM user_first_purchase
GROUP BY 1
ORDER BY 1
"""

df_new = client.query(query_new_customers).to_dataframe()
df_new['first_purchase_week'] = df_new['first_purchase_week'].astype(str)

In [39]:
colors = ['#378ADD' if week != '2019-11-17' else '#D85A30'
          for week in df_new['first_purchase_week']]

fig = go.Figure(go.Bar(
    x=df_new['first_purchase_week'],
    y=df_new['new_paying_customers'],
    text=df_new['new_paying_customers'].apply(lambda x: f'{x:,}'),
    textposition='outside',
    marker=dict(color=colors)  # ← 리스트로 색상 지정
))

fig.update_layout(
    title='주차별 신규 결제 고객 수',
    xaxis=dict(title='첫 구매 주차'),
    yaxis=dict(title='신규 결제 고객 수'),
    height=500
)

fig.show()

5. 카테고리별 판매량/수익(11월 3째주)

In [40]:
query_promo_category = """
SELECT
    COALESCE(category_code, '기타(unknown)') AS category,
    COUNT(*) AS sales_volume,
    ROUND(SUM(price), 2) AS total_revenue,
    COUNT(DISTINCT user_id) AS purchasing_users
FROM `gen-lang-client-0420777856.ecommerce_project.events`
WHERE event_type = 'purchase'
    AND DATE_TRUNC(DATE(event_time), WEEK) = '2019-11-17'
GROUP BY 1
ORDER BY sales_volume DESC
LIMIT 10
"""

df_promo_cat = client.query(query_promo_category).to_dataframe()

In [71]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 카테고리별 고정 색상 딕셔너리
color_map = {
    'electronics.smartphone':           '#378ADD',
    '기타(unknown)':                     '#73726c',
    'electronics.audio.headphone':      '#1D9E75',
    'electronics.video.tv':             '#D85A30',
    'electronics.clocks':               '#D4537E',
    'appliances.kitchen.washer':        '#7F77DD',
    'computers.notebook':               '#639922',
    'appliances.environment.vacuum':    '#BA7517',
    'appliances.kitchen.refrigerators': '#A32D2D',
    'apparel.shoes':                    '#888780'
}

# 데이터프레임 카테고리 순서에 맞게 색상 리스트 생성
colors_promo = [color_map.get(c, '#cccccc') for c in df_promo_cat['category']]

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=['판매량 비중', '총수익 비중']
)

total_sales   = df_promo_cat['sales_volume'].sum()
total_revenue = df_promo_cat['total_revenue'].sum()

# 왼쪽: 판매량
fig.add_trace(
    go.Pie(
        labels=df_promo_cat['category'],
        values=df_promo_cat['sales_volume'],
        hole=0.5,
        name='판매량',
        marker_colors=colors_promo,
        textinfo='label+percent',       # ← 라벨 + 퍼센트
        textposition='outside',         # ← 바깥에 표시
        hovertemplate='%{label}<br>판매량: %{value:,}건<br>비중: %{percent}'
    ),
    row=1, col=1
)

# 오른쪽: 수익
fig.add_trace(
    go.Pie(
        labels=df_promo_cat['category'],
        values=df_promo_cat['total_revenue'],
        hole=0.5,
        name='총수익',
        marker_colors=colors_promo,
        textinfo='label+percent',
        textposition='outside',
        hovertemplate='%{label}<br>수익: $%{value:,.0f}<br>비중: %{percent}'
    ),
    row=1, col=2
)

fig.update_layout(
    title='11월 3주차(11/17~11/23) 카테고리별 판매량 및 수익',
    height=500,
    # 가운데 총합 annotation
    showlegend=False,
    annotations=[
        dict(
            text=f'총 판매량<br><b>{total_sales:,}</b>',
            x=0.225, y=0.45,
            font=dict(size=13, color='darkblue'),
            showarrow=False
        ),
        dict(
            text=f'총 수익<br><b>${total_revenue/1000000:.1f}M</b>',
            x=0.772, y=0.45,
            font=dict(size=13, color='darkblue'),
            showarrow=False
        )
    ]
)


fig.show()

6. 카테고리별 판매량/수익(11월 전체)

In [60]:
query_monthly_cat = """
SELECT
    COALESCE(category_code, '기타(unknown)') AS category,
    COUNT(*) AS monthly_sales_volume,
    ROUND(SUM(price), 2) AS monthly_total_revenue,
    COUNT(DISTINCT user_id) AS monthly_purchasing_users
FROM `gen-lang-client-0420777856.ecommerce_project.events`
WHERE event_type = 'purchase'
GROUP BY 1
ORDER BY monthly_sales_volume DESC
LIMIT 10
"""

df_monthly_cat = client.query(query_monthly_cat).to_dataframe()


In [73]:
# 카테고리별 고정 색상 딕셔너리
color_map = {
    'electronics.smartphone':        '#378ADD',
    '기타(unknown)':                  '#73726c',
    'electronics.audio.headphone':   '#1D9E75',
    'electronics.video.tv':          '#D85A30',
    'electronics.clocks':            '#D4537E',
    'appliances.kitchen.washer':     '#7F77DD',
    'computers.notebook':            '#639922',
    'appliances.environment.vacuum': '#BA7517',
    'appliances.kitchen.refrigerators': '#A32D2D',
    'apparel.shoes':                 '#888780'
}

# 데이터프레임 카테고리 순서에 맞게 색상 리스트 생성
colors_monthly = [color_map.get(c, '#cccccc') for c in df_monthly_cat['category']]

total_sales   = df_monthly_cat['monthly_sales_volume'].sum()
total_revenue = df_monthly_cat['monthly_total_revenue'].sum()

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=['판매량 비중', '총수익 비중']
)

fig.add_trace(
    go.Pie(
        labels=df_monthly_cat['category'],
        values=df_monthly_cat['monthly_sales_volume'],
        hole=0.5,
        marker_colors=colors_monthly,
        textinfo='label+percent',
        textposition='outside',
        hovertemplate='%{label}<br>판매량: %{value:,}건<br>비중: %{percent}'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Pie(
        labels=df_monthly_cat['category'],
        values=df_monthly_cat['monthly_total_revenue'],
        hole=0.5,
        marker_colors=colors_monthly,
        textinfo='label+percent',
        textposition='outside',
        hovertemplate='%{label}<br>수익: $%{value:,.0f}<br>비중: %{percent}'
    ),
    row=1, col=2
)

fig.update_layout(
    title='11월 전체 카테고리별 판매량 및 수익',
    height=500,
    showlegend=False,
    annotations=[
        dict(
            text=f'총 판매량<br><b>{total_sales:,}</b>',
            x=0.225, y=0.45,
            font=dict(size=13, color='darkblue'),
            showarrow=False
        ),
        dict(
            text=f'총 수익<br><b>${total_revenue/1000000:.1f}M</b>',
            x=0.774, y=0.45,
            font=dict(size=13, color='darkblue'),
            showarrow=False
        )
    ]
)

fig.show()

7. 11월 3째주 브랜드별 판매량 **및수익**

In [81]:
query_promo_brand = """
SELECT
    COALESCE(brand, '기타(unknown)') AS brand,
    COUNT(*) AS sales_volume,
    ROUND(SUM(price), 2) AS total_revenue,
    COUNT(DISTINCT user_id) AS purchasing_users
FROM `gen-lang-client-0420777856.ecommerce_project.events`
WHERE event_type = 'purchase'
    AND DATE_TRUNC(DATE(event_time), WEEK) = '2019-11-17'
GROUP BY 1
ORDER BY sales_volume DESC
LIMIT 10
"""

df_promo_brand = client.query(query_promo_brand).to_dataframe()

In [82]:
brands = df_promo_brand['brand'].tolist()
sales  = df_promo_brand['sales_volume'].tolist()
rev    = df_promo_brand['total_revenue'].tolist()

view_to_cart  = round(sales[1] / sales[0] * 100, 2)
cart_to_purch = round(rev[1]   / rev[0]   * 100, 2)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=brands,
    y=sales,
    name='판매량',
    marker=dict(color='#378ADD'),
    yaxis='y1'
))

fig.add_trace(go.Scatter(
    x=brands,
    y=rev,
    name='총수익',
    mode='lines+markers',
    marker=dict(color='#D85A30', size=7),
    line=dict(color='#D85A30', width=2),
    yaxis='y2'
))

fig.update_layout(
    title='11월 3주차(11/17~11/23) 브랜드별 판매량 및 총수익',
    xaxis=dict(title='브랜드'),
    yaxis=dict(title='판매량', titlefont=dict(color='#378ADD'), tickfont=dict(color='#378ADD')),
    yaxis2=dict(
        title='총수익',
        titlefont=dict(color='#D85A30'),
        tickfont=dict(color='#D85A30'),
        overlaying='y',
        side='right',
        tickformat='$,.0f'
    ),
    legend=dict(x=0.8, y=1.1, orientation='h'),
    plot_bgcolor='white',
    height=500
)

fig.show()

8. 11월 전체 브랜드별 판매량 및수익

In [83]:
query_monthly_brand = """
SELECT
    COALESCE(brand, '기타(unknown)') AS brand,
    COUNT(*) AS monthly_sales_volume,
    ROUND(SUM(price), 2) AS monthly_total_revenue,
    COUNT(DISTINCT user_id) AS monthly_purchasing_users
FROM `gen-lang-client-0420777856.ecommerce_project.events`
WHERE event_type = 'purchase'
GROUP BY 1
ORDER BY monthly_sales_volume DESC
LIMIT 10
"""

df_monthly_brand = client.query(query_monthly_brand).to_dataframe()

In [84]:
brands = df_monthly_brand['brand'].tolist()
sales  = df_monthly_brand['monthly_sales_volume'].tolist()
rev    = df_monthly_brand['monthly_total_revenue'].tolist()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=brands,
    y=sales,
    name='판매량',
    marker=dict(color='#378ADD'),
    yaxis='y1'
))

fig.add_trace(go.Scatter(
    x=brands,
    y=rev,
    name='총수익',
    mode='lines+markers',
    marker=dict(color='#D85A30', size=7),
    line=dict(color='#D85A30', width=2),
    yaxis='y2'
))

fig.update_layout(
    title='11월 전체 브랜드별 판매량 vs 총수익',
    xaxis=dict(title='브랜드'),
    yaxis=dict(
        title='판매량',
        titlefont=dict(color='#378ADD'),
        tickfont=dict(color='#378ADD')
    ),
    yaxis2=dict(
        title='총수익',
        titlefont=dict(color='#D85A30'),
        tickfont=dict(color='#D85A30'),
        overlaying='y',
        side='right',
        tickformat='$,.0f'
    ),
    legend=dict(x=0.8, y=1.1, orientation='h'),
    plot_bgcolor='white',
    height=500
)

fig.show()